# Day 082 — Solution: An Agent That Remembers You

In [ ]:
_SRC = '"""memory_agent.py — Day 082: Agent Memory.\n\nDays 79-81 built agents that act, reason, and route across many tools - but every\none of them starts each task from a blank slate. This day gives an agent memory:\nshort-term working memory for the current conversation, and long-term memory that\npersists across sessions in SQLite, so the agent remembers you the next time.\n\nPieces:\n  safe_parse_json / call_llm   - reused (Day 79)\n  WorkingMemory                - short-term, in-session, bounded turn log\n  LongTermMemory               - durable key/value facts in SQLite (persists)\n  build_extraction_prompt / extract_memories - decide what is worth remembering\n  build_memory_prompt          - inject long-term profile + recent turns\n  MemoryAgent                  - an assistant that remembers you\n\nSetup:\n    pip install ollama\n    ollama pull llama3.2\n"""\nimport json\n\n# ── helpers reused from Day 79 ───────────────────────────────────────────────\ndef safe_parse_json(text):\n    """Slice first \'{\' to last \'}\' and parse. Returns dict|None (Day 79)."""\n    start, end = text.find("{"), text.rfind("}")\n    if start == -1 or end == -1 or end < start:\n        return None\n    try:\n        data = json.loads(text[start:end + 1])\n    except (json.JSONDecodeError, ValueError):\n        return None\n    return data if isinstance(data, dict) else None\n\n\ndef call_llm(messages, llm_fn=None):\n    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""\n    if llm_fn is not None:\n        return llm_fn(messages)\n    import ollama\n    resp = ollama.chat(model="llama3.2", messages=messages)\n    return resp["message"]["content"]\n\n# ── short-term working memory (this session only) ────────────────────────────\nclass WorkingMemory:\n    """Short-term memory: the recent turns of the current session.\n\n    Bounded to the last `max_turns` messages so the prompt never grows without\n    limit, and cleared at a session boundary. This is the agent\'s scratchpad -\n    it does NOT survive a restart.\n    """\n\n    def __init__(self, max_turns=10):\n        self.max_turns = max_turns\n        self._turns = []\n\n    def add(self, role, content):\n        """Append a {role, content} turn; keep only the last max_turns. Returns self."""\n        self._turns.append({"role": role, "content": str(content)})\n        if len(self._turns) > self.max_turns:\n            self._turns = self._turns[-self.max_turns:]\n        return self\n\n    def turns(self):\n        """Return a copy of the recent turns."""\n        return list(self._turns)\n\n    def render(self):\n        """Render the turns as text, one \'role: content\' line each."""\n        return "\\n".join(t["role"] + ": " + t["content"] for t in self._turns)\n\n    def clear(self):\n        """Forget the session (end-of-session boundary)."""\n        self._turns.clear()\n\n    def __len__(self):\n        return len(self._turns)\n\n# ── long-term memory (persists across sessions, SQLite) ──────────────────────\nimport sqlite3\n\n\nclass LongTermMemory:\n    """Durable key/value memory backed by SQLite - survives restarts.\n\n    Default db_path is ":memory:" (a private in-process database). Pass a file\n    path to persist across sessions: a new LongTermMemory on the same path sees\n    everything a previous one remembered.\n    """\n\n    def __init__(self, db_path=":memory:"):\n        self.db_path = db_path\n        self._conn = sqlite3.connect(db_path)\n        self._conn.execute(\n            "CREATE TABLE IF NOT EXISTS memories "\n            "(key TEXT PRIMARY KEY, value TEXT NOT NULL)"\n        )\n        self._conn.commit()\n\n    def remember(self, key, value):\n        """Store or overwrite a fact by key. Returns self."""\n        self._conn.execute(\n            "INSERT OR REPLACE INTO memories(key, value) VALUES(?, ?)",\n            (str(key), str(value)))\n        self._conn.commit()\n        return self\n\n    def recall(self, key):\n        """Return the stored value for a key, or None if unknown."""\n        row = self._conn.execute(\n            "SELECT value FROM memories WHERE key = ?", (str(key),)).fetchone()\n        return row[0] if row else None\n\n    def search(self, term):\n        """Return [{key, value}] where term (case-insensitive) is in key or value."""\n        term = str(term).lower()\n        rows = self._conn.execute("SELECT key, value FROM memories").fetchall()\n        return [{"key": k, "value": v} for k, v in rows\n                if term in k.lower() or term in v.lower()]\n\n    def all(self):\n        """Return every fact as [{key, value}], ordered by key."""\n        rows = self._conn.execute(\n            "SELECT key, value FROM memories ORDER BY key").fetchall()\n        return [{"key": k, "value": v} for k, v in rows]\n\n    def forget(self, key):\n        """Delete a fact by key. Returns self."""\n        self._conn.execute("DELETE FROM memories WHERE key = ?", (str(key),))\n        self._conn.commit()\n        return self\n\n    def __len__(self):\n        return self._conn.execute("SELECT COUNT(*) FROM memories").fetchone()[0]\n\n    def close(self):\n        """Close the database connection."""\n        self._conn.close()\n\n# ── deciding what is worth remembering (LLM-driven) ──────────────────────────\ndef build_extraction_prompt(message):\n    """Ask the model which durable facts about the user to store long-term."""\n    system = "\\n".join([\n        "You extract durable facts about the user that are worth remembering.",\n        "Return ONLY a JSON object mapping short snake_case keys to string values.",\n        "Store only stable facts - name, location, preferences, goals.",\n        "Ignore small talk and one-off questions.",\n        "If there is nothing worth remembering, return {}.",\n    ])\n    return [{"role": "system", "content": system},\n            {"role": "user", "content": str(message)}]\n\n\ndef extract_memories(message, llm_fn=None):\n    """Return durable facts to store as {key: value}. Never raises.\n\n    Unparseable model output falls back to an empty dict, so a bad extraction\n    simply stores nothing rather than crashing the turn.\n    """\n    response = call_llm(build_extraction_prompt(message), llm_fn=llm_fn)\n    data = safe_parse_json(response) or {}\n    return {str(k): str(v) for k, v in data.items()}\n\n# ── recall: injecting memory into the prompt ─────────────────────────────────\ndef build_memory_prompt(message, working, longterm):\n    """Build a chat prompt that injects long-term profile + short-term turns.\n\n    The system message carries what we durably know about the user; the recent\n    working-memory turns follow; the new message comes last.\n    """\n    facts = longterm.all()\n    profile = "\\n".join("- " + f["key"] + ": " + f["value"] for f in facts)\n    system = "\\n".join([\n        "You are a helpful assistant with memory of the user.",\n        "",\n        "What you remember about the user:",\n        profile if profile else "(nothing yet)",\n    ])\n    messages = [{"role": "system", "content": system}]\n    messages.extend(working.turns())\n    messages.append({"role": "user", "content": str(message)})\n    return messages\n\n# ── the agent that remembers you ─────────────────────────────────────────────\nclass MemoryAgent:\n    """An assistant that remembers you within a session and across sessions.\n\n    Two memories work together: a WorkingMemory for the current conversation\n    (short-term, cleared at a session boundary) and a LongTermMemory for durable\n    facts (persisted in SQLite). Each turn extracts new facts, injects everything\n    remembered into the prompt, answers, and records the exchange.\n\n    Example::\n\n        mem = LongTermMemory("user.db")          # persists across runs\n        agent = MemoryAgent(longterm=mem, llm_fn=my_llm_fn)\n        agent.chat("Hi, I\'m Kutlwano and I love SQL.")\n        agent.chat("What do you know about me?")\n    """\n\n    def __init__(self, longterm=None, llm_fn=None, max_turns=10):\n        self.longterm = longterm if longterm is not None else LongTermMemory()\n        self.working = WorkingMemory(max_turns=max_turns)\n        self._llm_fn = llm_fn\n\n    def chat(self, message):\n        """Remember new facts, answer with memory in context, record the turn."""\n        for key, value in extract_memories(message, llm_fn=self._llm_fn).items():\n            self.longterm.remember(key, value)\n        prompt = build_memory_prompt(message, self.working, self.longterm)\n        reply = call_llm(prompt, llm_fn=self._llm_fn)\n        self.working.add("user", message)\n        self.working.add("assistant", reply)\n        return reply\n\n    def remember(self, key, value):\n        """Store a durable fact directly. Returns self."""\n        self.longterm.remember(key, value)\n        return self\n\n    def recall(self, key):\n        """Look up a durable fact by key."""\n        return self.longterm.recall(key)\n\n    def profile(self):\n        """Everything remembered about the user, as [{key, value}]."""\n        return self.longterm.all()\n\n    def end_session(self):\n        """Clear short-term working memory; long-term persists."""\n        self.working.clear()\n'
from pathlib import Path
Path('memory_agent.py').write_text(_SRC, encoding='utf-8')
print('memory_agent.py written.')

In [ ]:

import os, tempfile, json
from memory_agent import (
    WorkingMemory, LongTermMemory, build_extraction_prompt, extract_memories,
    build_memory_prompt, MemoryAgent,
)

def _mock_llm(facts=None, reply='Sure!'):
    payload = json.dumps(facts or {})
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        return payload if 'extract' in system.lower() else reply
    return _fn

# 1. WorkingMemory
wm = WorkingMemory(max_turns=3)
wm.add('user', 'hi').add('assistant', 'hello')
assert len(wm) == 2 and wm.turns()[0] == {'role': 'user', 'content': 'hi'}
for i in range(5):
    wm.add('user', str(i))
assert len(wm) == 3 and wm.turns()[0]['content'] == '2'   # bounded
print("✅ WorkingMemory (bounded, copy, clear)")

# 2. LongTermMemory
lt = LongTermMemory()
lt.remember('name', 'Kutlwano').remember('lang', 'Python')
assert lt.recall('name') == 'Kutlwano' and lt.recall('missing') is None
lt.remember('name', 'Kutlwano M.')
assert lt.recall('name') == 'Kutlwano M.' and len(lt) == 2   # upsert
assert lt.search('python')[0]['key'] == 'lang'
lt.forget('lang'); assert lt.recall('lang') is None
path = tempfile.NamedTemporaryFile(suffix='.db', delete=False).name
try:
    a = LongTermMemory(path); a.remember('goal', 'ship agents'); a.close()
    assert LongTermMemory(path).recall('goal') == 'ship agents'   # persists
finally:
    os.unlink(path)
print("✅ LongTermMemory (SQLite upsert / search / forget / persistence)")

# 3. extract_memories
msgs = build_extraction_prompt('I am Kutlwano')
assert 'extract' in msgs[0]['content'].lower()
assert extract_memories('x', llm_fn=_mock_llm(facts={'name': 'K'})) == {'name': 'K'}
assert extract_memories('x', llm_fn=lambda m: 'no json') == {}       # never raises
print("✅ extract_memories (LLM-driven, safe fallback)")

# 4. build_memory_prompt
lt2 = LongTermMemory(); lt2.remember('name', 'Kutlwano')
p = build_memory_prompt('hello', WorkingMemory(), lt2)
assert p[0]['role'] == 'system' and 'name: Kutlwano' in p[0]['content']
assert p[-1] == {'role': 'user', 'content': 'hello'}
assert 'nothing yet' in build_memory_prompt('hi', WorkingMemory(), LongTermMemory())[0]['content']
print("✅ build_memory_prompt (profile in system, message last)")

# 5. MemoryAgent
agent = MemoryAgent(llm_fn=_mock_llm(facts={'name': 'Kutlwano'}, reply='Hi Kutlwano!'))
assert agent.chat('I am Kutlwano') == 'Hi Kutlwano!'
assert agent.recall('name') == 'Kutlwano' and len(agent.working) == 2
agent.remember('lang', 'Python')
assert {'key': 'lang', 'value': 'Python'} in agent.profile()
agent.end_session()
assert len(agent.working) == 0 and agent.recall('name') == 'Kutlwano'   # long-term kept
fresh = MemoryAgent(longterm=agent.longterm, llm_fn=_mock_llm(reply='ok'))
assert fresh.recall('name') == 'Kutlwano'   # a new agent still remembers you
print("✅ MemoryAgent (extract / recall / persist across sessions)")

print("\nAgent memory complete!")
